# 08 — Reviewer controls: $H_0$, PCA centering và PCA refit

Notebook này theo cùng workflow với `00_main_results.ipynb`: cấu hình tập trung, một job cho mỗi `(family, pair, arm, seed)`, teacher cache dùng chung, resume theo final-test record, và aggregate mean ± sample std.

Ba experiment family:

1. **H0 controls**: endpoint-only, endpoint+$H_0$, sorted spanning-path, teacher-MST, kNN-distribution và native-Gram controls; đây là family duy nhất được thay auxiliary objective, còn mọi setting khác giữ theo base.
2. **PCA preprocessing**: recipe hiện tại, textbook centered PCA, PCA whitening, uncentered SVD, và random orthonormal projection; mọi arm giữ nguyên best-base $H_0$ objective và epoch-wise Procrustes refit.
3. **PCA gauge/refit**: cùng một PCA subspace và best-base $H_0$ objective, chỉ thay orientation/update schedule: none, random orthogonal, fixed $R^{(0)}$, one mid-training refit, và epoch-wise Procrustes.

Các structural controls đọc cùng native teacher. `sorted_pairwise` dùng một spanning path độc lập với geometry, phủ mọi sample và có đúng $B-1$ scalar constraints như $H_0$; `teacher-MST` cũng có $B-1$, còn kNN dùng $k=1$ ($B$ directed distances). Tên output mới `endpoint_sorted_spanning_path` và `endpoint_native_gram_w1` buộc hai control đã sửa chạy mới thay vì resume artifact cũ.

> Dùng `EXECUTE` làm công tắc dry-run/execute. Nên kiểm tra commands và số jobs trước; `JOBS_PER_GPU = 1` nếu cần timing không bị nhiễu bởi GPU co-location.

In [2]:
# 1. Cấu hình thí nghiệm
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
PAIRS = {
    "qwen3_0.6b_to_minilm_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "teacher_pooling": "last_token",
        "min_vram_gib": 12,
    },
    "bge_m3_to_minilm_h768": {
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base",
        "teacher_pooling": "cls",
        "min_vram_gib": 12,
    },
    "qwen3_4b_to_bert_base": {
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "teacher_pooling": "last_token",
        "min_vram_gib": 24,
    },
}

# H0/PCA preprocessing đang screening trên configuration (c). Gauge/refit chạy
# configurations (a) và (c), đúng hai cột của bảng mechanism trong paper.
H0_PAIRS = ["qwen3_0.6b_to_minilm_h384"]
PCA_PAIRS = ["qwen3_0.6b_to_minilm_h384"]
GAUGE_PAIRS = ["qwen3_0.6b_to_minilm_h384", "qwen3_4b_to_bert_base"]
SEEDS = [42, 43, 44]
RANDOM_PROJECTION_DRAWS = [0, 1, 2]

TRAIN_DATA_REL = Path("data/train_set/merged_3_data_5k_each.csv")
MAX_LENGTH = 256
EPOCHS = 5
BATCH_SIZE = 128
LEARNING_RATE = 7e-5
GAUGE_SAMPLES = 16384
TOPO_BATCH_SIZE = 128
LAMBDA_H0 = 0.5
NUM_WORKERS = 2
PROBE_EVERY = 250
PROBE_SIZE = 1024

# Một process trên mỗi GPU cho timing có thể công bố. Tăng JOBS_PER_GPU chỉ
# để hoàn thành sweep nhanh hơn; notebook sẽ flag các rate metrics là co-located.
GPUS = ["0"]
JOBS_PER_GPU = 1
STOP_ON_ERROR = True
REQUIRE_ALL_RUNS = True
EVAL_RETRIEVAL = False
HELD_OUT_PROTOCOL = True
EXECUTE = True
INSTALL_REQUIREMENTS = True
UPDATE_REPO = False
SAVE_TO_GOOGLE_DRIVE = False

# Ba structural controls dùng cùng weight với H0 để giữ protocol matched.
# Gram dùng native teacher (trước PCA/gauge); weight không chọn bằng test result.
H0_ARMS = {
    "endpoint_only": ["--lambda_topo", "0", "--lambda_gram", "0"],
    "endpoint_h0": ["--lambda_topo", str(LAMBDA_H0), "--lambda_gram", "0", "--structural_loss", "h0"],
    "endpoint_sorted_spanning_path": ["--lambda_topo", str(LAMBDA_H0), "--lambda_gram", "0", "--structural_loss", "sorted_pairwise"],
    "endpoint_teacher_mst": ["--lambda_topo", str(LAMBDA_H0), "--lambda_gram", "0", "--structural_loss", "teacher_mst"],
    "endpoint_knn_distribution": ["--lambda_topo", str(LAMBDA_H0), "--lambda_gram", "0", "--structural_loss", "knn_distribution", "--structural_knn_k", "1"],
    "endpoint_native_gram_w1": ["--lambda_topo", "0", "--lambda_gram", "1"],
}

PCA_ARMS = {
    # Fit centered; apply without subtracting mean: paper recipe.
    "pca_current": ["--projection_type", "pca", "--pca_center_fit", "--no-pca_subtract_mean"],
    # Fit and apply after subtracting mean: textbook PCA coordinates.
    "pca_centered": ["--projection_type", "pca", "--pca_center_fit", "--pca_subtract_mean"],
    # Textbook whitening: centered fit, mean subtraction, inverse-sqrt spectrum.
    "pca_whitened": ["--projection_type", "pca_whiten", "--pca_center_fit", "--pca_subtract_mean"],
    # No centering at fit or application time.
    "svd_uncentered": ["--projection_type", "pca", "--no-pca_center_fit", "--no-pca_subtract_mean"],
}

# common_command không gắn gauge flags: mỗi family phải khai báo chúng tường minh.
PCA_CURRENT_FLAGS = ["--projection_type", "pca", "--pca_center_fit", "--no-pca_subtract_mean"]
PROCRUSTES_REFIT_FLAGS = [
    "--gauge_align", "--gauge_rotation", "procrustes",
    "--gauge_refit_every", "1", "--gauge_align_samples", str(GAUGE_SAMPLES),
]
BEST_BASE_OBJECTIVE_FLAGS = [
    "--lambda_topo", str(LAMBDA_H0), "--lambda_gram", "0", "--structural_loss", "h0",
]
ONE_REFIT_EVERY = (EPOCHS + 1) // 2
assert 0 < ONE_REFIT_EVERY < EPOCHS and 2 * ONE_REFIT_EVERY >= EPOCHS
GAUGE_ARMS = {
    "gauge_none": ["--no-gauge_align", "--gauge_refit_every", "0"],
    "gauge_random": ["--gauge_align", "--gauge_rotation", "random", "--gauge_random_seed", "0", "--gauge_refit_every", "0", "--gauge_align_samples", str(GAUGE_SAMPLES)],
    "gauge_fixed_r0": ["--gauge_align", "--gauge_rotation", "procrustes", "--gauge_refit_every", "0", "--gauge_align_samples", str(GAUGE_SAMPLES)],
    # Với EPOCHS=5, refit_every=3 cập nhật đúng một lần sau epoch 3.
    "gauge_one_refit": ["--gauge_align", "--gauge_rotation", "procrustes", "--gauge_refit_every", str(ONE_REFIT_EVERY), "--gauge_align_samples", str(GAUGE_SAMPLES)],
    "gauge_epochwise_refit": PROCRUSTES_REFIT_FLAGS,
}

RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
RUN_NAME_OVERRIDE = None  # điền tên run cũ để resume
RUN_NAME = RUN_NAME_OVERRIDE or f"review_h0_pca_controls_{RUN_STAMP}"

assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert set(H0_PAIRS).issubset(PAIRS) and set(PCA_PAIRS).issubset(PAIRS) and set(GAUGE_PAIRS).issubset(PAIRS)
assert JOBS_PER_GPU >= 1 and GPUS
print(f"Run: {RUN_NAME}")
print(f"H0 pairs: {H0_PAIRS}; arms: {list(H0_ARMS)}")
print(f"PCA pairs: {PCA_PAIRS}; deterministic arms: {list(PCA_ARMS)}; random draws: {RANDOM_PROJECTION_DRAWS}")
print(f"Gauge/refit pairs: {GAUGE_PAIRS}; arms: {list(GAUGE_ARMS)}")

Run: review_h0_pca_controls_20260906-104859
H0 pairs: ['qwen3_0.6b_to_minilm_h384']; arms: ['endpoint_only', 'endpoint_h0', 'endpoint_sorted_pairwise', 'endpoint_teacher_mst', 'endpoint_knn_distribution', 'endpoint_gram_w1', 'endpoint_gram_w10']
PCA pairs: ['qwen3_0.6b_to_minilm_h384']; deterministic arms: ['pca_current', 'pca_centered', 'pca_whitened', 'svd_uncentered']; random draws: [0, 1, 2]


In [3]:
# 2. Repo, dependencies, output và GPU
import os
import subprocess
import sys
import torch

cwd = Path.cwd().resolve()
if (cwd / "main.py").is_file() and (cwd / "distiller.py").is_file():
    PROJECT_DIR = cwd
else:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"

if UPDATE_REPO:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)

try:
    from google.colab import drive as colab_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"

RUN_ROOT = OUTPUT_BASE / RUN_NAME
CACHE_DIR = OUTPUT_BASE / "teacher_cache"
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu corpus: {TRAIN_DATA}"
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU runtime trước khi chạy notebook.")
if hasattr(torch.cuda, "is_bf16_supported") and not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU phải hỗ trợ BF16.")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
print(f"Project: {PROJECT_DIR}")
print(f"Corpus: {TRAIN_DATA}")
print(f"Output: {RUN_ROOT}")

cuda:0: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
Project: /content/embedding-kd
Corpus: /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv
Output: /content/embedding-kd/runs/review_h0_pca_controls_20260906-104859


In [4]:
# 3. Tạo unified plan cho H0, PCA preprocessing và PCA gauge/refit
import shlex

def run_dir(family, pair, arm, seed, draw=None):
    base = RUN_ROOT / family / pair / arm
    if draw is not None:
        base = base / f"draw_{draw}"
    return base / f"seed_{seed}"

def common_command(pair_name, output_dir, seed):
    pair = PAIRS[pair_name]
    command = [
        sys.executable, str(PROJECT_DIR / "main.py"),
        "--method", "geoode",
        "--train_data", str(TRAIN_DATA),
        "--student_model", pair["student"],
        "--teacher_model", pair["teacher"],
        "--teacher_pooling", pair["teacher_pooling"],
        "--student_pooling", "cls",
        "--batch_size", str(BATCH_SIZE),
        "--epochs", str(EPOCHS),
        "--save_every", str(EPOCHS),
        "--lr", str(LEARNING_RATE),
        "--max_length", str(MAX_LENGTH),
        "--seed", str(seed),
        "--num_workers", str(NUM_WORKERS),
        "--eval_every", "0",
        "--cache_dir", str(CACHE_DIR),
        "--save_dir", str(output_dir),
        "--lambda_end", "1",
        "--lambda_ctr", "0",
        "--lambda_h1", "0",
        "--topo_batch_size", str(TOPO_BATCH_SIZE),
        "--topo_teacher_source", "original",
        "--probe_every", str(PROBE_EVERY),
        "--probe_size", str(PROBE_SIZE),
        "--no_wandb",
    ]
    if HELD_OUT_PROTOCOL:
        command.extend(["--no-evaluate_test_each_epoch", "--pair_threshold_source", "validation"])
    else:
        command.extend(["--pair_threshold_source", "test"])
    if not EVAL_RETRIEVAL:
        command.append("--no_eval_retrieval")
    return command

JOBS = []
for pair_name in H0_PAIRS:
    for arm, flags in H0_ARMS.items():
        for seed in SEEDS:
            output_dir = run_dir("h0", pair_name, arm, seed)
            JOBS.append({
                "family": "h0", "pair": pair_name, "arm": arm,
                "draw": None, "seed": seed, "output_dir": output_dir,
                "command": common_command(pair_name, output_dir, seed)
                    + PCA_CURRENT_FLAGS + PROCRUSTES_REFIT_FLAGS + flags,
            })

for pair_name in PCA_PAIRS:
    for arm, flags in PCA_ARMS.items():
        for seed in SEEDS:
            output_dir = run_dir("pca", pair_name, arm, seed)
            JOBS.append({
                "family": "pca", "pair": pair_name, "arm": arm,
                "draw": None, "seed": seed, "output_dir": output_dir,
                "command": common_command(pair_name, output_dir, seed)
                    + PROCRUSTES_REFIT_FLAGS + BEST_BASE_OBJECTIVE_FLAGS + flags,
            })
    for draw in RANDOM_PROJECTION_DRAWS:
        arm = "random_procrustes"
        for seed in SEEDS:
            output_dir = run_dir("pca", pair_name, arm, seed, draw=draw)
            JOBS.append({
                "family": "pca", "pair": pair_name, "arm": arm,
                "draw": draw, "seed": seed, "output_dir": output_dir,
                "command": common_command(pair_name, output_dir, seed) + PROCRUSTES_REFIT_FLAGS + BEST_BASE_OBJECTIVE_FLAGS + [
                    "--projection_type", "random", "--projection_seed", str(draw),
                ],
            })

for pair_name in GAUGE_PAIRS:
    for arm, flags in GAUGE_ARMS.items():
        for seed in SEEDS:
            output_dir = run_dir("gauge", pair_name, arm, seed)
            JOBS.append({
                "family": "gauge", "pair": pair_name, "arm": arm,
                "draw": None, "seed": seed, "output_dir": output_dir,
                "command": common_command(pair_name, output_dir, seed)
                    + PCA_CURRENT_FLAGS + BEST_BASE_OBJECTIVE_FLAGS + flags,
            })

names = [(j["family"], j["pair"], j["arm"], j["draw"], j["seed"]) for j in JOBS]
assert len(names) == len(set(names)), "Plan có job trùng"
def flag_value(command, flag):
    return command[command.index(flag) + 1] if flag in command else None

# Protocol guards: PCA preprocessing phải giữ gauge refit + best-base H0;
# gauge family phải giữ PCA + best-base H0 trong khi chỉ đổi gauge schedule.
for job in JOBS:
    command = job["command"]
    if job["family"] in {"h0", "pca"}:
        assert flag_value(command, "--gauge_rotation") == "procrustes"
        assert flag_value(command, "--gauge_refit_every") == "1"
    if job["family"] in {"pca", "gauge"}:
        assert flag_value(command, "--lambda_topo") == str(LAMBDA_H0)
        assert flag_value(command, "--lambda_gram") == "0"
        assert flag_value(command, "--structural_loss") == "h0"
    if job["family"] == "gauge":
        assert flag_value(command, "--projection_type") == "pca"
    if job["family"] == "h0" and job["arm"] == "endpoint_native_gram_w1":
        assert flag_value(command, "--topo_teacher_source") == "original"
        assert flag_value(command, "--lambda_topo") == "0"
        assert flag_value(command, "--lambda_gram") == "1"
    if job["family"] == "h0" and job["arm"] == "endpoint_sorted_spanning_path":
        assert flag_value(command, "--topo_teacher_source") == "original"
        assert flag_value(command, "--lambda_topo") == str(LAMBDA_H0)
        assert flag_value(command, "--lambda_gram") == "0"
        assert flag_value(command, "--structural_loss") == "sorted_pairwise"
expected_refit_every = {
    "gauge_none": "0", "gauge_random": "0", "gauge_fixed_r0": "0",
    "gauge_one_refit": str(ONE_REFIT_EVERY), "gauge_epochwise_refit": "1",
}
for job in [item for item in JOBS if item["family"] == "gauge"]:
    assert flag_value(job["command"], "--gauge_refit_every") == expected_refit_every[job["arm"]]
print(f"Plan: {len(JOBS)} jobs")
for job in JOBS:
    label = f"{job['family']}/{job['pair']}/{job['arm']}"
    if job["draw"] is not None:
        label += f"/draw_{job['draw']}"
    print(f"[{label}/seed_{job['seed']}] {shlex.join(job['command'])}")

Plan: 42 jobs
[h0/qwen3_0.6b_to_minilm_h384/endpoint_only/seed_42] /usr/bin/python3 /content/embedding-kd/main.py --method geoode --train_data /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --teacher_pooling last_token --student_pooling cls --batch_size 128 --epochs 5 --save_every 5 --lr 7e-05 --max_length 256 --seed 42 --num_workers 2 --eval_every 0 --cache_dir /content/embedding-kd/runs/teacher_cache --save_dir /content/embedding-kd/runs/review_h0_pca_controls_20260906-104859/h0/qwen3_0.6b_to_minilm_h384/endpoint_only/seed_42 --lambda_end 1 --lambda_ctr 0 --lambda_h1 0 --topo_batch_size 128 --topo_teacher_source original --gauge_align --gauge_rotation procrustes --gauge_refit_every 1 --gauge_align_samples 16384 --probe_every 250 --probe_size 1024 --no_wandb --no-evaluate_test_each_epoch --pair_threshold_source validation --no_eval_retrieval --projection_type pc

In [5]:
# 4. Chạy/resume jobs
import json
import time

sys.path.insert(0, str(PROJECT_DIR))
from src.job_runner import gpu_slots, run_jobs_parallel  # noqa: E402

def last_json_record(path, predicate=None):
    if not path.is_file():
        return None
    found = None
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                if predicate is None or predicate(record):
                    found = record
    return found

def final_test_record(output_dir):
    return last_json_record(
        output_dir / "metrics.jsonl",
        lambda record: bool(record.get("test")) and not record.get("train"),
    )

def prewarm_cache(pair_name, gpu):
    pair = PAIRS[pair_name]
    command = [
        sys.executable, str(PROJECT_DIR / "main.py"), "--method", "geoode",
        "--train_data", str(TRAIN_DATA), "--student_model", pair["student"],
        "--teacher_model", pair["teacher"], "--teacher_pooling", pair["teacher_pooling"],
        "--cache_dir", str(CACHE_DIR), "--cache_only", "--no_eval_retrieval", "--no_wandb",
    ]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "WANDB_MODE": "disabled"}
    print(f"[CACHE {pair_name}] {shlex.join(command)}")
    subprocess.run(command, cwd=PROJECT_DIR, env=env, check=True)

pending, status = [], []
for job in JOBS:
    metrics = job["output_dir"] / "metrics.jsonl"
    if final_test_record(job["output_dir"]) is not None:
        status.append({**{k: job[k] for k in ("family", "pair", "arm", "draw", "seed")}, "status": "skipped_complete"})
    elif metrics.exists():
        raise RuntimeError(f"Run dở dang: {metrics}. Dùng RUN_NAME mới hoặc archive riêng run này.")
    else:
        job["output_dir"].mkdir(parents=True, exist_ok=True)
        job["name"] = f"{job['family']}/{job['pair']}/{job['arm']}/seed_{job['seed']}"
        job["log_path"] = job["output_dir"] / "train.log"
        pending.append(job)

print(f"Complete: {len(status)}; pending: {len(pending)}")
if not EXECUTE:
    print("Dry run: đổi EXECUTE=True trong cell cấu hình rồi chạy lại cell này.")
elif pending:
    # Warm cache trước fan-out để không charge cache lạnh cho một arm ngẫu nhiên.
    for index, pair_name in enumerate(sorted({job['pair'] for job in pending})):
        prewarm_cache(pair_name, GPUS[index % len(GPUS)])

    slots = gpu_slots(GPUS, JOBS_PER_GPU)
    def on_finish(job, row):
        complete = row["returncode"] == 0 and final_test_record(job["output_dir"]) is not None
        payload = {
            "status": "complete" if complete else "failed",
            "wall_seconds": row["seconds"],
            "max_parallel": JOBS_PER_GPU,
            "cuda_visible_devices": row["gpu"],
        }
        (job["output_dir"] / "runner_timing.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
        return {**row, **payload}

    rows = run_jobs_parallel(
        pending, cwd=PROJECT_DIR,
        env={**os.environ, "WANDB_MODE": "disabled", "TOKENIZERS_PARALLELISM": "false"},
        slots=slots, stop_on_error=STOP_ON_ERROR, poll_seconds=30.0, on_finish=on_finish,
    )
    status.extend(rows)
    (RUN_ROOT / "run_status.json").write_text(json.dumps(status, indent=2, default=str), encoding="utf-8")
    failed = [row for row in rows if row.get("status") == "failed"]
    if failed:
        raise RuntimeError(f"{len(failed)} jobs failed; xem train.log trong từng run directory.")

Complete: 0; pending: 42
[CACHE qwen3_0.6b_to_minilm_h384] /usr/bin/python3 /content/embedding-kd/main.py --method geoode --train_data /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --teacher_pooling last_token --cache_dir /content/embedding-kd/runs/teacher_cache --cache_only --no_eval_retrieval --no_wandb
[START 1/42] h0/qwen3_0.6b_to_minilm_h384/endpoint_only/seed_42 on GPU 0 (pid 3237) -> /content/embedding-kd/runs/review_h0_pca_controls_20260906-104859/h0/qwen3_0.6b_to_minilm_h384/endpoint_only/seed_42/train.log
[START 2/42] h0/qwen3_0.6b_to_minilm_h384/endpoint_only/seed_43 on GPU 0 (pid 3238) -> /content/embedding-kd/runs/review_h0_pca_controls_20260906-104859/h0/qwen3_0.6b_to_minilm_h384/endpoint_only/seed_43/train.log
[START 3/42] h0/qwen3_0.6b_to_minilm_h384/endpoint_only/seed_44 on GPU 0 (pid 3239) -> /content/embedding-kd/runs/review_h0_pca_controls_20

In [7]:
# 5. Collect, paired deltas, structural diagnostics và paper-ready tables
import numpy as np
import pandas as pd
from IPython.display import display

BENCHMARK_ORDER = ["banking77", "tweet", "emotion", "mrpc", "scitail", "wic", "sick", "sts12", "stsb"]
SUMMARY_ORDER = ["avg_iod", "avg_ood", "avg_all"]

def benchmark_name(path):
    name = Path(path).stem
    return name[:-5] if name.endswith("_test") else name

def score_from_payload(family, raw):
    if family == "classification": return float(raw["f1"])
    if family == "pair": return float(raw["average_precision"])
    if family == "sts": return float(raw)
    raise KeyError(family)

def read_projection(output_dir):
    path = output_dir / "teacher_projection.pt"
    if not path.is_file(): return {}
    saved = torch.load(path, map_location="cpu", weights_only=False)
    gauge = saved.get("gauge_stats") or {}
    history = saved.get("gauge_history") or []
    final_gauge = history[-1] if history else gauge
    return {
        "projection_type": saved.get("projection_type"),
        "pca_center_fit": saved.get("pca_center_fit"),
        "pca_subtract_mean": saved.get("pca_subtract_mean"),
        "explained_energy": saved.get("explained_energy"),
        "gauge_rotation": saved.get("gauge_rotation") if saved.get("gauge_align") else "none",
        "gauge_refit_every": saved.get("gauge_refit_every"),
        "gauge_updates": max(0, len(history) - 1),
        "gauge_cos_before": gauge.get("cos_before"),
        "gauge_cos_after": gauge.get("cos_after"),
        "gauge_final_cos_after": final_gauge.get("cos_after"),
        "gauge_final_cos_previous": final_gauge.get("cos_previous_gauge"),
        "target_participation_ratio": gauge.get("participation_ratio"),
    }

rows, missing = [], []
for job in JOBS:
    final = final_test_record(job["output_dir"])
    if final is None:
        missing.append(job)
        continue
    payload = final["test"]
    row = {k: job[k] for k in ("family", "pair", "arm", "draw", "seed")}
    for family in ("classification", "pair", "sts"):
        for path, raw in payload.get(family, {}).items():
            row[benchmark_name(path)] = score_from_payload(family, raw)
    for key in SUMMARY_ORDER:
        row[key] = float(payload["summary"][key])
    row["avg_all"] = float(np.mean([row[name] for name in BENCHMARK_ORDER]))
    probe = last_json_record(job["output_dir"] / "probe_metrics.jsonl") or {}
    for key in ("probe_gram_rmse_teacher", "probe_gram_corr_teacher", "probe_knn_overlap_teacher", "probe_mutual_knn_teacher", "probe_h0_w1_teacher"):
        row[key] = probe.get(key)
    row.update(read_projection(job["output_dir"]))
    timing_path = job["output_dir"] / "runner_timing.json"
    timing = json.loads(timing_path.read_text(encoding="utf-8")) if timing_path.is_file() else {}
    row["wall_minutes"] = timing.get("wall_seconds", np.nan) / 60 if timing.get("wall_seconds") is not None else np.nan
    row["co_located"] = int(timing.get("max_parallel", 1) or 1) > 1
    rows.append(row)

if missing:
    print(f"[MISSING] {len(missing)}/{len(JOBS)} runs")
    for job in missing[:20]: print(job["family"], job["pair"], job["arm"], job["draw"], job["seed"])
    if REQUIRE_ALL_RUNS: raise RuntimeError("Chưa aggregate vì thiếu run.")

by_seed = pd.DataFrame(rows).sort_values(["family", "pair", "arm", "draw", "seed"], na_position="first")
assert not by_seed.empty, "Không có final-test result"
by_seed.to_csv(RUN_ROOT / "review_controls_by_seed.csv", index=False)

projection_metric_columns = ["explained_energy", "gauge_refit_every", "gauge_updates", "gauge_cos_before", "gauge_cos_after", "gauge_final_cos_after", "gauge_final_cos_previous", "target_participation_ratio"]
metric_columns = [*BENCHMARK_ORDER, *SUMMARY_ORDER, "probe_gram_rmse_teacher", "probe_gram_corr_teacher", "probe_knn_overlap_teacher", "probe_mutual_knn_teacher", "probe_h0_w1_teacher", *projection_metric_columns, "wall_minutes"]
# Một diagnostic có thể vắng ở run cũ; vẫn tạo cột NaN để schema của bảng ổn định.
for column in metric_columns:
    if column not in by_seed.columns: by_seed[column] = np.nan
summary = by_seed.groupby(["family", "pair", "arm", "draw"], dropna=False, sort=False)[metric_columns].agg(["mean", "std", "count"])
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index()
summary.to_csv(RUN_ROOT / "review_controls_mean_std.csv", index=False)

# Paired H0/Gram deltas against endpoint-only, matched by pair and training seed.
h0 = by_seed[by_seed.family == "h0"]
baseline = h0[h0.arm == "endpoint_only"].set_index(["pair", "seed"])
delta_rows = []
for arm in [name for name in H0_ARMS if name != "endpoint_only"]:
    candidate = h0[h0.arm == arm].set_index(["pair", "seed"])
    for index in baseline.index.intersection(candidate.index):
        row = {"pair": index[0], "seed": index[1], "arm": arm}
        for metric in [*BENCHMARK_ORDER, *SUMMARY_ORDER]:
            row[f"delta_{metric}"] = candidate.loc[index, metric] - baseline.loc[index, metric]
        delta_rows.append(row)
h0_deltas = pd.DataFrame(delta_rows)
h0_deltas.to_csv(RUN_ROOT / "h0_paired_deltas.csv", index=False)
h0_delta_summary = h0_deltas.groupby(["pair", "arm"], sort=False).agg({"delta_avg_iod": ["mean", "std"], "delta_avg_ood": ["mean", "std"], "delta_avg_all": ["mean", "std"]})
h0_delta_summary.columns = [f"{metric}_{stat}" for metric, stat in h0_delta_summary.columns]
h0_delta_summary = h0_delta_summary.reset_index()
h0_delta_summary.to_csv(RUN_ROOT / "h0_paired_delta_summary.csv", index=False)

# PCA deltas against current recipe. Random draws remain separate; do not pool
# nine seed×draw rows as if they were nine independent training seeds.
pca = by_seed[by_seed.family == "pca"]
pca_base = pca[pca.arm == "pca_current"].set_index(["pair", "seed"])
pca_delta_rows = []
for candidate_row in pca[pca.arm != "pca_current"].itertuples():
    index = (candidate_row.pair, candidate_row.seed)
    if index not in pca_base.index: continue
    out = {"pair": candidate_row.pair, "arm": candidate_row.arm, "draw": candidate_row.draw, "seed": candidate_row.seed}
    for metric in [*BENCHMARK_ORDER, *SUMMARY_ORDER]:
        out[f"delta_{metric}"] = getattr(candidate_row, metric) - pca_base.loc[index, metric]
    pca_delta_rows.append(out)
pca_deltas = pd.DataFrame(pca_delta_rows)
pca_deltas.to_csv(RUN_ROOT / "pca_paired_deltas.csv", index=False)

# Gauge/refit deltas against no gauge, paired by configuration and seed.
# Every row retains the same best-base H0 objective (lambda_topo=LAMBDA_H0).
gauge_runs = by_seed[by_seed.family == "gauge"]
gauge_base = gauge_runs[gauge_runs.arm == "gauge_none"].set_index(["pair", "seed"])
gauge_delta_rows = []
for candidate_row in gauge_runs[gauge_runs.arm != "gauge_none"].itertuples():
    index = (candidate_row.pair, candidate_row.seed)
    if index not in gauge_base.index: continue
    out = {"pair": candidate_row.pair, "arm": candidate_row.arm, "seed": candidate_row.seed}
    for metric in [*BENCHMARK_ORDER, *SUMMARY_ORDER]:
        out[f"delta_{metric}"] = getattr(candidate_row, metric) - gauge_base.loc[index, metric]
    gauge_delta_rows.append(out)
gauge_deltas = pd.DataFrame(gauge_delta_rows)
gauge_deltas.to_csv(RUN_ROOT / "gauge_paired_deltas.csv", index=False)
gauge_delta_summary = gauge_deltas.groupby(["pair", "arm"], sort=False).agg({"delta_avg_iod": ["mean", "std"], "delta_avg_ood": ["mean", "std"], "delta_avg_all": ["mean", "std"]})
gauge_delta_summary.columns = [f"{metric}_{stat}" for metric, stat in gauge_delta_summary.columns]
gauge_delta_summary = gauge_delta_summary.reset_index()
gauge_delta_summary.to_csv(RUN_ROOT / "gauge_paired_delta_summary.csv", index=False)

print("H0/Gram — mean ± sd by pair")
display(summary[summary.family == "h0"][["pair", "arm", "avg_iod_mean", "avg_iod_std", "avg_ood_mean", "avg_ood_std", "avg_all_mean", "avg_all_std", "wall_minutes_mean"]])
print("Paired deltas versus endpoint-only")
display(h0_delta_summary)
print("PCA — each deterministic arm / random draw")
display(summary[summary.family == "pca"][["pair", "arm", "draw", "avg_iod_mean", "avg_iod_std", "avg_ood_mean", "avg_ood_std", "avg_all_mean", "avg_all_std", "explained_energy_mean", "probe_gram_rmse_teacher_mean", "probe_knn_overlap_teacher_mean"]])
print("PCA gauge/refit — same PCA subspace and best-base H0 objective")
display(summary[summary.family == "gauge"][["pair", "arm", "avg_iod_mean", "avg_iod_std", "avg_ood_mean", "avg_ood_std", "avg_all_mean", "avg_all_std", "gauge_updates_mean", "gauge_cos_after_mean", "gauge_final_cos_after_mean"]])
print("Gauge/refit paired deltas versus no gauge")
display(gauge_delta_summary)
print(f"Saved tables to {RUN_ROOT}")

H0/Gram — mean ± sd by pair


,pair,arm,avg_iod_mean,avg_iod_std,avg_ood_mean,avg_ood_std,avg_all_mean,avg_all_std,wall_minutes_mean
0,qwen3_0.6b_to_minilm_h384,endpoint_gram_w1,0.674216,0.001373,0.766221,0.000384,0.735552,0.000374,1.022539
1,qwen3_0.6b_to_minilm_h384,endpoint_gram_w10,0.652458,0.000960,0.747601,0.000544,0.715887,0.000668,1.117046
2,qwen3_0.6b_to_minilm_h384,endpoint_h0,0.687070,0.000548,0.778598,0.000380,0.748089,0.000421,2.672863
3,qwen3_0.6b_to_minilm_h384,endpoint_knn_distribution,0.687630,0.000763,0.781550,0.000460,0.750243,0.000066,1.061438
4,qwen3_0.6b_to_minilm_h384,endpoint_only,0.676274,0.000720,0.775911,0.000413,0.742699,0.000182,2.872875
5,qwen3_0.6b_to_minilm_h384,endpoint_sorted_pairwise,0.672881,0.000348,0.767471,0.000843,0.735941,0.000674,2.456175
6,qwen3_0.6b_to_minilm_h384,endpoint_teacher_mst,0.685289,0.001000,0.779830,0.000685,0.748316,0.000762,2.378468


Paired deltas versus endpoint-only


,pair,arm,delta_avg_iod_mean,delta_avg_iod_std,delta_avg_ood_mean,delta_avg_ood_std,delta_avg_all_mean,delta_avg_all_std
0,qwen3_0.6b_to_minilm_h384,endpoint_h0,0.010796,0.001265,0.002687,0.000033,0.005390,0.000407
1,qwen3_0.6b_to_minilm_h384,endpoint_sorted_pairwise,-0.003393,0.001065,-0.008440,0.000459,-0.006758,0.000659
2,qwen3_0.6b_to_minilm_h384,endpoint_teacher_mst,0.009015,0.001647,0.003919,0.000439,0.005617,0.000764
3,qwen3_0.6b_to_minilm_h384,endpoint_knn_distribution,0.011356,0.000450,0.005639,0.000486,0.007544,0.000196
4,qwen3_0.6b_to_minilm_h384,endpoint_gram_w1,-0.002058,0.001699,-0.009691,0.000067,-0.007147,0.000529
5,qwen3_0.6b_to_minilm_h384,endpoint_gram_w10,-0.023816,0.001678,-0.028310,0.000165,-0.026812,0.000669


PCA — each deterministic arm / random draw


,pair,arm,draw,avg_iod_mean,avg_iod_std,avg_ood_mean,avg_ood_std,avg_all_mean,avg_all_std,explained_energy_mean,probe_gram_rmse_teacher_mean,probe_knn_overlap_teacher_mean
7,qwen3_0.6b_to_minilm_h384,pca_centered,NaN,0.668457,0.001495,0.749452,0.000214,0.722453,0.000566,0.928403,0.351588,0.422754
8,qwen3_0.6b_to_minilm_h384,pca_current,NaN,0.676364,0.000615,0.775532,0.000440,0.742476,0.000187,0.928403,0.186357,0.450358
9,qwen3_0.6b_to_minilm_h384,pca_whitened,NaN,0.681387,0.003304,0.758263,0.000817,0.732638,0.001646,0.928403,0.329567,0.368848
10,qwen3_0.6b_to_minilm_h384,random_procrustes,0.0,0.673791,0.000707,0.772792,0.000641,0.739792,0.000592,0.381220,0.204312,0.438151
11,qwen3_0.6b_to_minilm_h384,random_procrustes,1.0,0.674109,0.001699,0.774673,0.000472,0.741151,0.000358,0.375775,0.203564,0.435872
12,qwen3_0.6b_to_minilm_h384,random_procrustes,2.0,0.674836,0.000695,0.769484,0.000752,0.737935,0.000310,0.374257,0.183386,0.431120
13,qwen3_0.6b_to_minilm_h384,svd_uncentered,NaN,0.675916,0.002170,0.776106,0.000131,0.742709,0.000789,0.945318,0.198487,0.450293


Saved tables to /content/embedding-kd/runs/review_h0_pca_controls_20260906-104859


## Cách dùng kết quả trong paper

- `h0_paired_delta_summary.csv`: bảng compact cho main text, báo cáo $\Delta$ IOD/OOD/All của $H_0$ và Gram controls trên từng configuration.
- `review_controls_mean_std.csv`: full mean/std và structural diagnostics cho appendix.
- `pca_paired_deltas.csv`: paired difference giữa từng PCA variant và recipe hiện tại; tất cả đều giữ best-base $H_0$ objective.
- `gauge_paired_delta_summary.csv`: paired $\Delta$ IOD/OOD/All của random, fixed $R^{(0)}$, one-refit và epoch-wise refit so với no gauge, trên configurations (a) và (c).
- `review_controls_by_seed.csv` lưu cả `gauge_updates`, cosine của fit ban đầu và cosine sau refit cuối để audit đúng schedule.
- Random projection phải được đọc theo từng `draw`; trước hết lấy mean qua training seeds trong mỗi draw, sau đó mới mô tả spread giữa draws.

Teacher structural targets được dựng trong collate worker khi `topo_teacher_source=original`; whitening là projection arm riêng và luôn bật `pca_subtract_mean` trong protocol này. PCA/gauge controls đều giữ $\lambda_{H_0}=0.5$; chỉ structural-control family được thay auxiliary loss. `gauge_one_refit` dùng `refit_every=ceil(EPOCHS/2)`, nên với 5 epochs có đúng một update sau epoch 3; epoch-wise refit có 4 updates.